In [ ]:
# 本文件用于将日度数据表转化成月度数据表
%run config_0.ipynb
%run data_io_1.ipynb
%run clean_and_universe_2.ipynb
%run features_3.ipynb

import pandas as pd
import numpy as np

In [ ]:
# 把月度面板分成两个部分：滞后一期的收益率（体现因子的预测性）和形成期的因子特征
# 第一部分：滞后一期的收益率
def get_ym_monthly_excessret(df):
    df[form] = df[date].dt.to_period('M')
    # 收益要落在“收益所在月”（持有月），不要 +1；否则会和当期因子对齐，而不是预测下期
    df[ym] = df[form]

    ret = (
        df.groupby([ym , stock_code])[daily_ret]
        .apply(lambda x:(1 + x).prod() - 1)
        .rename(monthly_return)
    )
    rf = (
        df.groupby([ym , stock_code])[daily_rfret]
        .apply(lambda x:(1 + x).prod() - 1)
        .rename(monthly_rfreturn)
    )
    
    monthly = pd.concat([ret , rf] , axis = 1).reset_index()
    monthly[monthly_excess_ret] = monthly[monthly_return] - monthly[monthly_rfreturn]
    monthly = monthly[[stock_code , ym , monthly_return , monthly_rfreturn , monthly_excess_ret]]
    return monthly

In [ ]:
# 第二部分：形成期的因子特征
def get_form_signal(df):
    df[form] = df[date].dt.to_period('M')
    df[ym] = df[form] + 1

    # 每只股票每个形成月的最后一个交易日
    last_day = df.groupby([stock_code , form])[date].transform('max')

    print('查看last_day')
    print(last_day.head())
    print(last_day.shape)
    # 只保留每月最后一个交易日的因子值
    snap = df.loc[df[date] == last_day].copy()
    print('查看用date去匹配last_day的结果')
    print(snap.head())
    print(snap.shape)
    # 日度因子列 -> 月度因子列
    snap = snap.rename(columns = {
        day_market_value : monthly_market_value , 
        day_bm : monthly_bm , 
        day_mom : monthly_mom , 
        day_ep : monthly_ep , 
        day_cp : monthly_cp ,
        day_asset_growth : monthly_asset_growth , 
        day_volatility : monthly_volatility , 
        day_max : monthly_max , 
        day_illiquidity : monthly_illiquidity , 
        day_turnover_12m : monthly_turnover_12m , 
        day_abn_turnover_1m : monthly_abn_turnover_1m
    })

    # 只保留合并需要的列
    snap = snap[[stock_code , 
                 form , 
                 ym , 
                 monthly_market_value , 
                 monthly_bm , 
                 monthly_mom , 
                 monthly_ep , 
                 monthly_cp , 
                 monthly_asset_growth , 
                 monthly_volatility , 
                 monthly_max , 
                 monthly_illiquidity , 
                 monthly_turnover_12m , 
                 monthly_abn_turnover_1m]]
    
    return snap

In [ ]:
# 合并
def get_monthly_panel(monthly , df):
    monthly_final = monthly.merge(
        df , 
        left_on = [stock_code , ym] , 
        right_on = [stock_code , ym] , 
        how = 'left'
    )
    return monthly_final

df = read_daily_data()
df = cleaning(df)
df = universe(df)
cf_df = read_cf_data()
cf_df = cf_cleaning(cf_df)
bs_df = read_bs_data()
bs_df = bs_cleaning(bs_df)
df = get_market_feature(df)
df = get_momentum(df)
df = get_cp(df , cf_df)
df = get_asset_growth(df , bs_df)
df = get_excess_return(df)
df = get_bm(df)
df = get_ep(df)
df = get_one_month_volatility(df)
df = get_max_return(df)
df = get_illiquidity(df)
df = get_turnover_12m(df)
df = get_abn_turnover_1m(df)

snap = get_form_signal(df)
monthly = get_ym_monthly_excessret(df)
monthly_final = get_monthly_panel(monthly , snap)
print('查看月末截面快照')
print(snap.head())
print(snap.shape)
print('查看月度面板')
print(monthly.head())
print(monthly.shape)
print('查看最终月度面板')
print(monthly_final.head())
print(monthly_final.shape)